# FuelGap: Explainable Machine Learning for Urban Fuel-Use Gap Estimation and EV Fleet Prioritization

**Project title:** FuelGap: An Explainable, Leakage-Aware Machine Learning Framework for Urban Fuel-Use Gap Estimation and EV Fleet Conversion Prioritization

**Dataset:** Fuel Consumption 2000–2022  
**Dataset page:** https://www.kaggle.com/datasets/ahmettyilmazz/fuel-consumption

## Important research note
UFUG (Urban Fuel-Use Gap) is defined as the difference between certified city and highway fuel-consumption ratings, expressed in gal/100 mi. It is a rating-derived proxy, not a direct measurement of idling, congestion-related waste, real-world fuel savings, or emissions avoided through electrification.

This notebook compares feature settings, evaluates regression performance, explains model behavior, and builds a scenario-based EV prioritization tool. It does not claim a new machine-learning algorithm.


## Update History
Update 5 --> Based on chat suggestions(Leakage fixed). Previous code commented : New Best Model is RF with good R2

Update 6 --> Outlier Handling(capped, not removed) : R2 reduced significantly.

update 7 --> Outlier placement fixed. SMAPE added removing  mape : R2 reduced

update 8 --> Suggestions to increase the R2 value(Mode added,) : R2 improved

update 9 --> SHAP fixed for random forest only.

update 10 --> Updated naming issues, error curve etc.

## 1. Install Required Libraries

In [ ]:
!pip install -q kagglehub xgboost lightgbm catboost shap lime openpyxl

## 2. Import Libraries

In [ ]:
import os
import re
import glob
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.inspection import permutation_importance

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge, HuberRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR, LinearSVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    AdaBoostRegressor,
    BaggingRegressor,
    VotingRegressor,
    StackingRegressor
)
from sklearn.neural_network import MLPRegressor

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully.")

## 3. Load Dataset

This cell first tries to download the dataset directly from Kaggle using `kagglehub`.

If Kaggle download does not work in your Colab, set `USE_KAGGLEHUB = False` and upload the CSV manually.

In [ ]:
'''
USE_KAGGLEHUB = False
KAGGLE_DATASET_SLUG = "ahmettyilmazz/fuel-consumption"

def load_dataset_from_folder(folder_path):
    csv_files = glob.glob(os.path.join(folder_path, "**", "*.csv"), recursive=True)
    xlsx_files = glob.glob(os.path.join(folder_path, "**", "*.xlsx"), recursive=True)

    all_files = csv_files + xlsx_files
    if not all_files:
        raise FileNotFoundError("No CSV or Excel file found in the dataset folder.")

    # Choose the largest file because Kaggle dataset usually has one main CSV.
    main_file = max(all_files, key=os.path.getsize)
    print("Loaded file:", main_file)

    if main_file.lower().endswith(".csv"):
        return pd.read_csv(main_file)
    else:
        return pd.read_excel(main_file)

if USE_KAGGLEHUB:
    try:
        import kagglehub
        dataset_path = kagglehub.dataset_download(KAGGLE_DATASET_SLUG)
        df = load_dataset_from_folder(dataset_path)
    except Exception as e:
        print("Kagglehub download failed.")
        print("Error:", e)
        print("Please upload the CSV file manually.")
        from google.colab import files
        uploaded = files.upload()
        uploaded_file = list(uploaded.keys())[0]
        if uploaded_file.lower().endswith(".csv"):
            df = pd.read_csv(uploaded_file)
        else:
            df = pd.read_excel(uploaded_file)
else:
    from google.colab import files
    uploaded = files.upload()
    uploaded_file = list(uploaded.keys())[0]
    if uploaded_file.lower().endswith(".csv"):
        df = pd.read_csv(uploaded_file)
    else:
        df = pd.read_excel(uploaded_file)

print("Dataset shape:", df.shape)
display(df.head())
'''

In [ ]:
df=pd.read_csv("Fuel_Consumption_2000-2022.csv")
print("Dataset shape:", df.shape)
df.head()

## 4. Basic Dataset Inspection

In [ ]:
print("Shape:", df.shape)
print("\nColumn names:")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False).head(20))

print("\nDuplicate rows:", df.duplicated().sum())

display(df.describe(include="all").T.head(30))

## 5. Clean Column Names and Detect Important Columns

The code automatically detects city and highway fuel-consumption columns.

It supports both:
- **L/100 km** style columns, common in Canadian fuel-consumption datasets
- **MPG** style columns, common in US/EPA-style datasets

In [ ]:
original_columns = list(df.columns)

def normalize_col_name(col):
    col = str(col).strip().lower()
    col = col.replace("₂", "2")
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col).strip("_")
    return col

clean_cols = [normalize_col_name(c) for c in df.columns]
col_map = dict(zip(clean_cols, df.columns))

df_clean = df.copy()
df_clean.columns = clean_cols

print("Cleaned column names:")
for c in df_clean.columns:
    print(c)

def find_first_col(patterns, exclude_patterns=None):
    exclude_patterns = exclude_patterns or []
    for col in df_clean.columns:
        col_text = col.lower()
        if all(re.search(p, col_text) for p in patterns):
            if not any(re.search(ep, col_text) for ep in exclude_patterns):
                return col
    return None

# Possible city/highway columns
# Manual mapping for this dataset
city_col = "fuel_consumption" if "fuel_consumption" in df_clean.columns else None
hwy_col = "hwy_l_100_km" if "hwy_l_100_km" in df_clean.columns else None

print("Detected city fuel column:", city_col)
print("Detected highway fuel column:", hwy_col)

if city_col is None or hwy_col is None:
    raise ValueError("Could not detect city/highway fuel-consumption columns. Please check dataset columns manually.")

# If many columns contain city/highway, prefer ones that look like fuel consumption/rating.
# Manual mapping for this dataset
city_col = "fuel_consumption"
hwy_col = "hwy_l_100_km"

print("Detected city fuel column:", city_col)
print("Detected highway fuel column:", hwy_col)

if city_col is None or hwy_col is None:
    raise ValueError("Could not detect city/highway fuel-consumption columns. Please check dataset columns manually.")

print("Detected city fuel column:", city_col)
print("Detected highway fuel column:", hwy_col)

if city_col is None or hwy_col is None:
    raise ValueError("Could not detect city/highway fuel-consumption columns. Please check dataset columns manually.")

# Detect some useful columns

co2_col = find_first_col([r"(co2|emission)"], []) #co2_col = find_first_col([r"co2"], [])
engine_col = find_first_col([r"engine"], [])
cylinder_col = find_first_col([r"cylinder"], [])
#fuel_type_col = find_first_col([r"fuel", r"type"], [])
fuel_type_col = find_first_col([r"fuel", r"type"], [])
if fuel_type_col is None and "fuel" in df_clean.columns:
    fuel_type_col = "fuel"
transmission_col = find_first_col([r"transmission"], [])
vehicle_class_col = find_first_col([r"vehicle", r"class"], [])
make_col = find_first_col([r"make"], [])
model_col = find_first_col([r"model"], [])
year_col = find_first_col([r"(year|model_year)"], [])

important_cols = {
    "city_col": city_col,
    "hwy_col": hwy_col,
    "co2_col": co2_col,
    "engine_col": engine_col,
    "cylinder_col": cylinder_col,
    "fuel_type_col": fuel_type_col,
    "transmission_col": transmission_col,
    "vehicle_class_col": vehicle_class_col,
    "make_col": make_col,
    "model_col": model_col,
    "year_col": year_col
}
important_cols

## Select Option

In [ ]:
# ============================================================
# Experiment Setting
# ============================================================

EXPERIMENT_MODE = "inventory_assisted" # inventory_assisted or strict
# Options:
# "strict" = excludes model identity
# "inventory_assisted" = includes grouped model identity

INCLUDE_MODEL_GROUPED = EXPERIMENT_MODE == "inventory_assisted"

## 6. Feature Engineering: Urban Fuel-Use Gap (UFUG)

UFUG is defined as the difference between certified city and highway fuel-consumption ratings, converted to gal/100 mi. It is not a direct measure of idling or congestion-related fuel waste.

### If the dataset uses L/100 km
\[
UFUG = (CityFuelConsumption - HighwayFuelConsumption) \times 0.425143707
\]

### If the dataset uses MPG
\[
UFUG = \frac{100}{CityMPG} - \frac{100}{HighwayMPG}
\]

The target is **`ufug`**, measured in gal/100 mi.


In [ ]:
data = df_clean.copy()

# Convert detected city/highway columns to numeric if needed.
data[city_col] = pd.to_numeric(data[city_col], errors="coerce")
data[hwy_col] = pd.to_numeric(data[hwy_col], errors="coerce")

# Detect unit style from column names.
city_name = city_col.lower()
hwy_name = hwy_col.lower()

uses_mpg = ("mpg" in city_name) or ("mpg" in hwy_name)
uses_l100 = ("l_100" in city_name) or ("l_100" in hwy_name) or ("l_100km" in city_name) or ("l_100km" in hwy_name)

# If columns do not clearly state MPG but values are mostly below 25, this is probably L/100 km.
# If values are mostly above 25, this is probably MPG.
city_median = data[city_col].median()
hwy_median = data[hwy_col].median()

if not uses_mpg and not uses_l100:
    if city_median > 25 and hwy_median > 25:
        uses_mpg = True
    else:
        uses_l100 = True

L100KM_TO_GAL_PER_100MI = 0.425143707

if uses_mpg:
    print("Detected unit style: MPG")
    data["city_gal_per_100mi"] = 100 / data[city_col]
    data["hwy_gal_per_100mi"] = 100 / data[hwy_col]
    data["ufug"] = data["city_gal_per_100mi"] - data["hwy_gal_per_100mi"]
elif uses_l100:
    print("Detected unit style: L/100 km")
    data["city_gal_per_100mi"] = data[city_col] * L100KM_TO_GAL_PER_100MI
    data["hwy_gal_per_100mi"] = data[hwy_col] * L100KM_TO_GAL_PER_100MI
    data["ufug"] = (data[city_col] - data[hwy_col]) * L100KM_TO_GAL_PER_100MI
else:
    raise ValueError("Could not determine fuel-consumption unit style.")

# Remove impossible values.
data = data.replace([np.inf, -np.inf], np.nan)
data = data.dropna(subset=["ufug", "city_gal_per_100mi", "hwy_gal_per_100mi"])
data = data[data["ufug"] >= 0]

print("Data shape after target creation:", data.shape)
display(data[[
    city_col, hwy_col,
    "city_gal_per_100mi", "hwy_gal_per_100mi",
    "ufug"
]].head())

print("\nTarget summary:")
display(data["ufug"].describe())

## 7. Additional Feature Engineering

The following features help the model learn vehicle-level UFUG behavior from technical attributes.

In [ ]:
# Engine/cylinder ratio if both columns exist
if engine_col and cylinder_col:
    data[engine_col] = pd.to_numeric(data[engine_col], errors="coerce")
    data[cylinder_col] = pd.to_numeric(data[cylinder_col], errors="coerce")
    data["engine_per_cylinder"] = data[engine_col] / data[cylinder_col].replace(0, np.nan)

# Vehicle age from model year if available
if year_col:
    data[year_col] = pd.to_numeric(data[year_col], errors="coerce")
    data["vehicle_age_from_2022"] = 2022 - data[year_col]

# Transmission simplification
if transmission_col:
    data["transmission_family"] = data[transmission_col].astype(str).str.extract(r"([A-Za-z]+)", expand=False).fillna("Unknown")

# Fuel type simplification
if fuel_type_col:
    fuel_map = {
        "X": "Regular Gasoline",
        "Z": "Premium Gasoline",
        "D": "Diesel",
        "E": "Ethanol/E85",
        "N": "Natural Gas"
    }
    data["fuel_type_readable"] = data[fuel_type_col].astype(str).map(fuel_map).fillna(data[fuel_type_col].astype(str))

display(data.head())

In [ ]:
# ============================================================
# Additional Vehicle-Attribute Feature Engineering
# ============================================================

# Engine interaction features
if engine_col and cylinder_col:
    data[engine_col] = pd.to_numeric(data[engine_col], errors="coerce")
    data[cylinder_col] = pd.to_numeric(data[cylinder_col], errors="coerce")

    data["engine_size_x_cylinders"] = data[engine_col] * data[cylinder_col]
    data["engine_size_squared"] = data[engine_col] ** 2
    data["cylinders_squared"] = data[cylinder_col] ** 2

    data["engine_size_per_cylinder_v2"] = (
        data[engine_col] / data[cylinder_col].replace(0, np.nan)
    )

# Engine size category
if engine_col:
    data["engine_size_category"] = pd.cut(
        data[engine_col],
        bins=[0, 1.6, 2.5, 3.5, 5.0, np.inf],
        labels=["small", "compact", "medium", "large", "very_large"]
    ).astype(str)

# Cylinder category
if cylinder_col:
    data["cylinder_category"] = pd.cut(
        data[cylinder_col],
        bins=[0, 4, 6, 8, np.inf],
        labels=["low", "medium", "high", "very_high"]
    ).astype(str)

# Gear count from transmission text
if transmission_col:
    data["gear_count"] = (
        data[transmission_col]
        .astype(str)
        .str.extract(r"(\d+)")
        .astype(float)
    )

    data["has_gear_count"] = data["gear_count"].notna().astype(int)
    data["gear_count"] = data["gear_count"].fillna(data["gear_count"].median())

# Luxury or performance make indicator
if make_col:
    luxury_makes = [
        "BMW", "MERCEDES-BENZ", "AUDI", "PORSCHE", "JAGUAR",
        "LAND ROVER", "LEXUS", "INFINITI", "CADILLAC",
        "LINCOLN", "MASERATI", "BENTLEY", "ROLLS-ROYCE",
        "FERRARI", "LAMBORGHINI", "BUGATTI", "ASTON MARTIN"
    ]

    data["is_luxury_or_performance_make"] = (
        data[make_col].astype(str).str.upper().isin(luxury_makes).astype(int)
    )

# Common fleet make indicator
if make_col:
    fleet_makes = [
        "FORD", "CHEVROLET", "GMC", "RAM", "DODGE",
        "TOYOTA", "NISSAN", "HONDA", "MERCEDES-BENZ"
    ]

    data["is_common_fleet_make"] = (
        data[make_col].astype(str).str.upper().isin(fleet_makes).astype(int)
    )

# Vehicle age group
if "vehicle_age_from_2022" in data.columns:
    data["vehicle_age_group"] = pd.cut(
        data["vehicle_age_from_2022"],
        bins=[-1, 3, 7, 12, 20, np.inf],
        labels=["new", "recent", "middle", "old", "very_old"]
    ).astype(str)

# Group model names safely for inventory-assisted experiment
if model_col and model_col in data.columns:
    MODEL_MIN_COUNT = 5
    model_counts = data[model_col].astype(str).value_counts()
    common_models = model_counts[model_counts >= MODEL_MIN_COUNT].index

    data["vehicle_model_group"] = np.where(
        data[model_col].astype(str).isin(common_models),
        data[model_col].astype(str),
        "Other_Model"
    )

print("Additional feature engineering completed.")
display(data.head())

## 8. Exploratory Data Analysis(EDA)

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(data["ufug"], bins=30)
plt.xlabel("Urban Fuel-Use Gap (UFUG; gal/100 mi)")
plt.ylabel("Frequency")
plt.title("Distribution of Urban Fuel-Use Gap (UFUG)")
plt.show()

numeric_cols_for_corr = data.select_dtypes(include=[np.number]).columns.tolist()
corr = data[numeric_cols_for_corr].corr(numeric_only=True)

if len(numeric_cols_for_corr) > 1:
    plt.figure(figsize=(8, 6))
    plt.imshow(corr, aspect="auto")
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.index)), corr.index)
    plt.colorbar()
    plt.title("Numerical Feature Correlation Matrix")
    plt.tight_layout()
    plt.show()

In [ ]:
TARGET = "ufug"

##Outlier check and handling

In [ ]:
# ============================================================
# Visual Outlier Checking Before Removal or Capping
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Select numeric columns only
numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()

print("Numeric columns available for outlier visualization:")
print(numeric_cols)

# Optional: remove ID-like columns if they exist
id_like_cols = [
    col for col in numeric_cols
    if "id" in col.lower() or "year" == col.lower()
]

outlier_visual_cols = [col for col in numeric_cols if col not in id_like_cols]

print("\nNumeric columns used for outlier visualization:")
print(outlier_visual_cols)

In [ ]:
# ============================================================
# Boxplots for Numeric Columns
# ============================================================

for col in outlier_visual_cols:
    plt.figure(figsize=(8, 4))
    plt.boxplot(data[col].dropna(), vert=False)
    plt.title(f"Boxplot of {col}")
    plt.xlabel(col)
    plt.grid(axis="x", alpha=0.3)
    plt.show()

In [ ]:
# ============================================================
# Histograms for Numeric Columns
# ============================================================

for col in outlier_visual_cols:
    plt.figure(figsize=(8, 4))
    plt.hist(data[col].dropna(), bins=30)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.grid(axis="y", alpha=0.3)
    plt.show()

In [ ]:
# ============================================================
# IQR-Based Outlier Count Summary
# ============================================================

outlier_summary = []

for col in outlier_visual_cols:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_mask = (data[col] < lower_bound) | (data[col] > upper_bound)
    outlier_count = outlier_mask.sum()
    outlier_percentage = (outlier_count / len(data)) * 100

    outlier_summary.append({
        "Column": col,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower_Bound": lower_bound,
        "Upper_Bound": upper_bound,
        "Outlier_Count": outlier_count,
        "Outlier_Percentage": outlier_percentage
    })

outlier_summary_df = pd.DataFrame(outlier_summary)

outlier_summary_df = outlier_summary_df.sort_values(
    by="Outlier_Count",
    ascending=False
)

outlier_summary_df

In [ ]:
# ============================================================
# Columns Having Outliers
# ============================================================

columns_with_outliers = outlier_summary_df[outlier_summary_df["Outlier_Count"] > 0]

print("Columns with detected outliers:")
display(columns_with_outliers)

In [ ]:
# ============================================================
# Boxplots Only for Columns with Outliers
# ============================================================

outlier_cols = columns_with_outliers["Column"].tolist()

for col in outlier_cols:
    plt.figure(figsize=(8, 4))
    plt.boxplot(data[col].dropna(), vert=False)
    plt.title(f"Outlier Boxplot of {col}")
    plt.xlabel(col)
    plt.grid(axis="x", alpha=0.3)
    plt.show()

In [ ]:
# IMPORTANT: Do not calculate or apply outlier thresholds before splitting.
# This preserves the untouched test distribution. Any target cap is fitted
# using y_train only in the split/preprocessing cell below.
data_before_outlier_handling = data.copy()
data_raw_for_ranking = data.copy()
print("Outlier capping deferred until after the train/test split.")
print("Rows retained:", len(data))


In [ ]:
# ============================================================
# Boxplots After Outlier Handling
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Select numeric columns after outlier handling
numeric_cols_after = data.select_dtypes(include=[np.number]).columns.tolist()

# Remove ID-like or year columns from visualization
id_like_cols_after = [
    col for col in numeric_cols_after
    if "id" in col.lower() or col.lower() == "year"
]

boxplot_cols_after = [
    col for col in numeric_cols_after
    if col not in id_like_cols_after
]

print("Numeric columns used for boxplot after outlier handling:")
print(boxplot_cols_after)

for col in boxplot_cols_after:
    plt.figure(figsize=(8, 4))
    plt.boxplot(data[col].dropna(), vert=False)
    plt.title(f"Boxplot After Outlier Handling: {col}")
    plt.xlabel(col)
    plt.grid(axis="x", alpha=0.3)
    plt.show()

In [ ]:
# ============================================================
# Before vs After Outlier Handling Boxplots
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

numeric_cols_compare = data.select_dtypes(include=[np.number]).columns.tolist()

id_like_cols_compare = [
    col for col in numeric_cols_compare
    if "id" in col.lower() or col.lower() == "year"
]

compare_cols = [
    col for col in numeric_cols_compare
    if col not in id_like_cols_compare
]

for col in compare_cols:
    if col in data_before_outlier_handling.columns:
        plt.figure(figsize=(10, 4))

        plt.boxplot(
            [
                data_before_outlier_handling[col].dropna(),
                data[col].dropna()
            ],
            labels=["Before", "After"],
            vert=False
        )

        plt.title(f"Before vs After Outlier Handling: {col}")
        plt.xlabel(col)
        plt.grid(axis="x", alpha=0.3)
        plt.show()

## 9. Select Features and Target

### Very Important
Because the target is derived from city and highway fuel-consumption columns, the model **excludes** city/highway/combined consumption columns by default.  
This avoids data leakage and makes the research scientifically stronger.

If city/highway columns are included, the model may get very high accuracy, but the result will be less meaningful.

In [ ]:
# Copied and updated the previous code
# Jate real code safe thake.
TARGET = "ufug"

# Keep this True for the research paper.
LEAKAGE_SAFE_MODE = True

# Columns that should not be used as input features because they directly create or strongly leak the target.
target_source_cols = {
    city_col, hwy_col,
    "city_gal_per_100mi", "hwy_gal_per_100mi",
    TARGET
}

# Combined fuel and MPG columns are also often derived from city/highway values.
strict_leakage_exact_cols = {
    "fuel_consumption",
    "hwy_l_100_km",
    "comb_l_100_km",
    "comb_mpg",
    "emissions",
    "co2",
    "co2_emissions",
    "city_gal_per_100mi",
    "hwy_gal_per_100mi",
    TARGET
}

strict_leakage_patterns = [
    r"(^|_)city(_|$)",
    r"(^|_)hwy(_|$)",
    r"(^|_)highway(_|$)",
    r"(^|_)comb(_|$)",
    r"(^|_)combined(_|$)",
    r"(^|_)mpg(_|$)",
    r"fuel_consumption",
    r"l_100_km",
    r"gal_per_100mi",
    r"co2",
    r"emission"
]

id_like_keywords = ["vin", "id", "url", "link"]

drop_cols = set(target_source_cols)

if LEAKAGE_SAFE_MODE:
    for col in data.columns:
        if col in strict_leakage_exact_cols:
            drop_cols.add(col)
        if any(re.search(pattern, col) for pattern in strict_leakage_patterns):
            drop_cols.add(col)
        if any(kw == col or col.endswith("_" + kw) for kw in id_like_keywords):
            drop_cols.add(col)

# Keep original model name out if it has too many unique values.
# Make can be useful, but Model usually creates high-cardinality noise.
for col in data.columns:
    if data[col].dtype == "object":
        # Protect grouped model column in inventory-assisted experiment
        if INCLUDE_MODEL_GROUPED and col == "vehicle_model_group":
            continue
        unique_ratio = data[col].nunique(dropna=True) / max(len(data), 1)
        if unique_ratio > 0.50:
            drop_cols.add(col)

# Also avoid using model text column if detected.
# Always drop original raw model text because it can be too high-cardinality.
if model_col:
    drop_cols.add(model_col)

# Include or exclude grouped model identity depending on experiment mode.
if "vehicle_model_group" in data.columns and not INCLUDE_MODEL_GROUPED:
    drop_cols.add("vehicle_model_group")

feature_cols = [c for c in data.columns if c not in drop_cols]

X = data[feature_cols].copy()
y = data[TARGET].copy()

# Remove columns that are entirely missing.
X = X.dropna(axis=1, how="all")

print("Target:", TARGET)
print("Number of features:", X.shape[1])
print("\nSelected feature columns:")
for c in X.columns:
    print("-", c)

print("\nDropped columns:")
for c in sorted(drop_cols):
    if c in data.columns:
        print("-", c)

display(X.head())

In [ ]:
'''
TARGET = "ufug"

# Keep this True for the research paper.
LEAKAGE_SAFE_MODE = True

# Columns that should not be used as input features because they directly create or strongly leak the target.
target_source_cols = {
    city_col, hwy_col,
    "city_gal_per_100mi", "hwy_gal_per_100mi",
    TARGET
}

# Combined fuel and MPG columns are also often derived from city/highway values.
leakage_keywords = [
    "fuel_consumption", "consumption_city", "consumption_hwy", "consumption_highway",
    "fuel_consumption_comb", "comb_mpg", "combined", "mpg", "co2_emissions", "co2"
]

id_like_keywords = ["vin", "id", "url", "link"]

drop_cols = set(target_source_cols)

if LEAKAGE_SAFE_MODE:
    for col in data.columns:
        if any(kw in col for kw in leakage_keywords):
            drop_cols.add(col)
        if any(kw == col or col.endswith("_" + kw) for kw in id_like_keywords):
            drop_cols.add(col)

# Keep original model name out if it has too many unique values.
# Make can be useful, but Model usually creates high-cardinality noise.
for col in data.columns:
    if data[col].dtype == "object":
        unique_ratio = data[col].nunique(dropna=True) / max(len(data), 1)
        if unique_ratio > 0.50:
            drop_cols.add(col)

# Also avoid using model text column if detected.
if model_col:
    drop_cols.add(model_col)

feature_cols = [c for c in data.columns if c not in drop_cols]

X = data[feature_cols].copy()
y = data[TARGET].copy()

# Remove columns that are entirely missing.
X = X.dropna(axis=1, how="all")

print("Target:", TARGET)
print("Number of features:", X.shape[1])
print("\nSelected feature columns:")
for c in X.columns:
    print("-", c)

print("\nDropped columns:")
for c in sorted(drop_cols):
    if c in data.columns:
        print("-", c)

display(X.head())
'''

## 10. Train-Test Split and Preprocessing Pipeline

In [ ]:
# Split first. All learned preprocessing and target-cap thresholds are based on training data only.
from sklearn.model_selection import GroupShuffleSplit

TARGET = "ufug"
TEST_SIZE = 0.20
TARGET_CAP_IQR = True

X_train, X_test, y_train_raw, y_test_raw = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

# Cap the target using training-set IQR thresholds only, then apply the same bounds to train and test.
if TARGET_CAP_IQR:
    q1, q3 = y_train_raw.quantile(0.25), y_train_raw.quantile(0.75)
    iqr = q3 - q1
    target_lower_cap = max(0.0, q1 - 1.5 * iqr)
    target_upper_cap = q3 + 1.5 * iqr
    y_train = y_train_raw.clip(lower=target_lower_cap, upper=target_upper_cap)
    y_test = y_test_raw.clip(lower=target_lower_cap, upper=target_upper_cap)
else:
    target_lower_cap, target_upper_cap = None, None
    y_train, y_test = y_train_raw.copy(), y_test_raw.copy()

numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)
print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Target metrics will be reported on the " + ("training-IQR-capped" if TARGET_CAP_IQR else "uncapped") + " UFUG target.")

def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_transformer_scaled = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
numeric_transformer_tree = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", make_one_hot_encoder())
])
preprocessor_scaled = ColumnTransformer(transformers=[
    ("num", numeric_transformer_scaled, numeric_features),
    ("cat", categorical_transformer, categorical_features)
], remainder="drop")
preprocessor_tree = ColumnTransformer(transformers=[
    ("num", numeric_transformer_tree, numeric_features),
    ("cat", categorical_transformer, categorical_features)
], remainder="drop")
print("Preprocessing pipelines are ready.")


## 11. Define ML Regression Models

This section includes the important regression models for the paper:

- Baseline
- Linear models
- Regularized models
- KNN
- SVM
- Decision Tree
- Random Forest
- Extra Trees
- Gradient Boosting
- AdaBoost
- Bagging
- XGBoost
- LightGBM
- CatBoost
- MLP Neural Network
- Voting Ensemble
- Stacking Ensemble

In [ ]:
# Optional models
try:
    from xgboost import XGBRegressor
    xgboost_available = True
except Exception:
    xgboost_available = False

try:
    from lightgbm import LGBMRegressor
    lightgbm_available = True
except Exception:
    lightgbm_available = False

try:
    from catboost import CatBoostRegressor
    catboost_available = True
except Exception:
    catboost_available = False

models = {}

# Baseline
models["Dummy_Mean_Baseline"] = (preprocessor_scaled, DummyRegressor(strategy="mean"))

# Linear and regularized models
models["Linear_Regression"] = (preprocessor_scaled, LinearRegression())
models["Ridge"] = (preprocessor_scaled, Ridge(alpha=1.0, random_state=RANDOM_STATE))
models["Lasso"] = (preprocessor_scaled, Lasso(alpha=0.001, random_state=RANDOM_STATE, max_iter=10000))
models["ElasticNet"] = (preprocessor_scaled, ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=RANDOM_STATE, max_iter=10000))
models["Bayesian_Ridge"] = (preprocessor_scaled, BayesianRidge())
models["Huber_Regressor"] = (preprocessor_scaled, HuberRegressor(max_iter=1000))

# Distance/kernel models
models["KNN_Regressor"] = (preprocessor_scaled, KNeighborsRegressor(n_neighbors=5))
models["Linear_SVR"] = (preprocessor_scaled, LinearSVR(random_state=RANDOM_STATE, max_iter=10000))
models["SVR_RBF"] = (preprocessor_scaled, SVR(kernel="rbf", C=10, epsilon=0.01))

# Tree-based models
models["Decision_Tree"] = (preprocessor_tree, DecisionTreeRegressor(random_state=RANDOM_STATE, max_depth=None))
models["Random_Forest"] = (preprocessor_tree, RandomForestRegressor(
    n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1
))
models["Extra_Trees"] = (preprocessor_tree, ExtraTreesRegressor(
    n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1
))
models["Gradient_Boosting"] = (preprocessor_tree, GradientBoostingRegressor(random_state=RANDOM_STATE))
models["Hist_Gradient_Boosting"] = (preprocessor_tree, HistGradientBoostingRegressor(random_state=RANDOM_STATE))
models["AdaBoost"] = (preprocessor_tree, AdaBoostRegressor(
    estimator=DecisionTreeRegressor(max_depth=6, random_state=RANDOM_STATE),
    n_estimators=200,
    random_state=RANDOM_STATE
))
models["Bagging_Trees"] = (preprocessor_tree, BaggingRegressor(
    estimator=DecisionTreeRegressor(random_state=RANDOM_STATE),
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1
))

# Boosting libraries
if xgboost_available:
    models["XGBoost"] = (preprocessor_tree, XGBRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))

if lightgbm_available:
    models["LightGBM"] = (preprocessor_tree, LGBMRegressor(
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=31,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1
    ))


if catboost_available:
    models["CatBoost"] = (preprocessor_tree, CatBoostRegressor(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        loss_function="RMSE",
        random_seed=RANDOM_STATE,
        verbose=False
    ))
'''
# Used to increase R2 value
if catboost_available:
    models["CatBoost"] = (preprocessor_tree, CatBoostRegressor(
        iterations=1000,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=3,
        loss_function="RMSE",
        random_seed=RANDOM_STATE,
        verbose=False
    ))
    '''
# Neural network
models["MLP_Regressor"] = (preprocessor_scaled, MLPRegressor(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    alpha=0.001,
    learning_rate_init=0.001,
    max_iter=800,
    random_state=RANDOM_STATE
))

print("Number of base models:", len(models))
print(list(models.keys()))

## 12. Train, Cross-Validate, and Evaluate All Base Models

In [ ]:
def smape_score(y_true, y_pred, epsilon=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    valid = denominator > epsilon

    if valid.sum() == 0:
        return np.nan

    return np.mean(np.abs(y_true[valid] - y_pred[valid]) / denominator[valid]) * 100


def regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    smape = smape_score(y_true, y_pred)
    return mae, rmse, r2, smape

In [ ]:
# Model selection uses training-only cross-validation. The test set is evaluated only after selection.
from sklearn.model_selection import KFold, cross_validate

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2"
}
cv_rows = []
cv_fitted = {}

for name, (prep, estimator) in models.items():
    print(f"Cross-validating: {name}")
    pipe = Pipeline(steps=[("preprocessor", prep), ("model", estimator)])
    try:
        scores = cross_validate(
            pipe, X_train, y_train, cv=cv, scoring=scoring,
            n_jobs=1, error_score=np.nan, return_train_score=True
        )
        cv_rows.append({
            "Model": name,
            "CV_MAE": -float(np.nanmean(scores["test_MAE"])),
            "CV_RMSE": -float(np.nanmean(scores["test_RMSE"])),
            "CV_R2": float(np.nanmean(scores["test_R2"])),
            "Train_CV_RMSE": -float(np.nanmean(scores["train_RMSE"]))
        })
    except Exception as exc:
        print(f"CV failed for {name}: {exc}")

cv_results_df = pd.DataFrame(cv_rows).dropna(subset=["CV_RMSE"]).sort_values("CV_RMSE")
if cv_results_df.empty:
    raise RuntimeError("All model cross-validation runs failed; no model can be selected.")
cv_results_df.to_csv("cross_validation_model_selection.csv", index=False)
best_model_name = cv_results_df.iloc[0]["Model"]
best_model = Pipeline(steps=[
    ("preprocessor", models[best_model_name][0]),
    ("model", models[best_model_name][1])
])
best_model.fit(X_train, y_train)
best_test_pred = best_model.predict(X_test)

def smape_score(y_true, y_pred, epsilon=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    den = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    valid = den > epsilon
    return float(np.mean(np.abs(y_true[valid] - y_pred[valid]) / den[valid]) * 100) if valid.any() else np.nan

def regression_metrics(y_true, y_pred):
    return (
        mean_absolute_error(y_true, y_pred),
        np.sqrt(mean_squared_error(y_true, y_pred)),
        r2_score(y_true, y_pred),
        smape_score(y_true, y_pred)
    )

mae, rmse, r2, smape = regression_metrics(y_test, best_test_pred)
results_df = pd.DataFrame([{
    "Model": best_model_name,
    "Test_MAE": mae,
    "Test_RMSE": rmse,
    "Test_R2": r2,
    "Test_SMAPE": smape,
    "Target_Capped": TARGET_CAP_IQR,
    "Target_Cap_Lower": target_lower_cap,
    "Target_Cap_Upper": target_upper_cap
}])
results_df.to_csv("selected_model_test_metrics.csv", index=False)
print("Selected model by training CV:", best_model_name)
print("Untouched test-set metrics (test target uses training-derived cap if enabled):")
display(results_df)


In [ ]:
# Cross-validation results used for model selection (training data only).
display(cv_results_df.head(10))
print("Saved: cross_validation_model_selection.csv")


## Tuned Random Forest and Extra Trees

In [ ]:
'''
# ============================================================
# Hyperparameter Tuning for Strong Tree-Based Models
# ============================================================

from sklearn.model_selection import RandomizedSearchCV

RUN_TUNING = True
N_ITER_TUNING = 8   # Increase to 15 or 20 for final run

tuned_results = {}
tuned_rows = []

if RUN_TUNING:

    tuning_configs = {
        "Random_Forest_Tuned": {
            "base_model": RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
            "params": {
                "model__n_estimators": [300, 500, 700],
                "model__max_depth": [None, 20, 30, 40],
                "model__min_samples_split": [2, 5, 10],
                "model__min_samples_leaf": [1, 2, 4],
                "model__max_features": ["sqrt", "log2", 0.7]
            }
        },
        "Extra_Trees_Tuned": {
            "base_model": ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=-1),
            "params": {
                "model__n_estimators": [300, 500, 700],
                "model__max_depth": [None, 20, 30, 40],
                "model__min_samples_split": [2, 5, 10],
                "model__min_samples_leaf": [1, 2, 4],
                "model__max_features": ["sqrt", "log2", 0.7]
            }
        }
    }

    for tuned_name, config in tuning_configs.items():
        print("\nTuning:", tuned_name)

        tuned_pipe = Pipeline(steps=[
            ("preprocessor", preprocessor_tree),
            ("model", config["base_model"])
        ])

        search = RandomizedSearchCV(
            estimator=tuned_pipe,
            param_distributions=config["params"],
            n_iter=N_ITER_TUNING,
            scoring="neg_root_mean_squared_error",
            cv=3,
            verbose=2,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

        search.fit(X_train, y_train)

        best_tuned_model = search.best_estimator_
        tuned_pred = best_tuned_model.predict(X_test)

        mae, rmse, r2, smape = regression_metrics(y_test, tuned_pred)

        tuned_results[tuned_name] = best_tuned_model
        trained_pipelines[tuned_name] = best_tuned_model

        tuned_rows.append({
            "Model": tuned_name,
            "Test_MAE": mae,
            "Test_RMSE": rmse,
            "Test_R2": r2,
            "Test_SMAPE": smape
        })

        print("Best CV RMSE:", -search.best_score_)
        print("Best params:", search.best_params_)
        print("Test R2:", r2)

    tuned_results_df = pd.DataFrame(tuned_rows)

    results_df = pd.concat([results_df, tuned_results_df], ignore_index=True)
    results_df = results_df.sort_values(by="Test_RMSE", ascending=True).reset_index(drop=True)

    display(results_df)
    '''

## 13. Visualize Model Comparison

In [ ]:
if not results_df.empty:
    plt.figure(figsize=(10, 6))
    plt.barh(results_df["Model"], results_df["Test_RMSE"])
    plt.xlabel("Test RMSE")
    plt.ylabel("Model")
    plt.title("Model Comparison by Test RMSE")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 6))
    plt.barh(results_df["Model"], results_df["Test_R2"])
    plt.xlabel("Test R²")
    plt.ylabel("Model")
    plt.title("Model Comparison by Test R²")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

## 14. Build Ensemble Models

The ensemble models use the strongest available tree/boosting models.

- **Voting Regressor:** averages predictions from strong models.
- **Stacking Regressor:** learns how to combine multiple models using a final meta-model.

In [ ]:
'''
# ============================================================
# 14. Build Faster Ensemble Model
# ============================================================

from sklearn.ensemble import VotingRegressor
from sklearn.pipeline import Pipeline
import pandas as pd
import numpy as np

# Select ensemble base models from available strong learners
ensemble_candidate_names = [
    "Random_Forest",
    "Extra_Trees",
    "Gradient_Boosting",
    "XGBoost",
    "LightGBM",
    "CatBoost"
]

available_candidates = [name for name in ensemble_candidate_names if name in models]
print("Available ensemble candidates:", available_candidates)

# Select only top 3 models based on previous Test_RMSE
top_available_candidates = [
    name for name in results_df["Model"].tolist()
    if name in available_candidates
][:3]

print("Selected ensemble candidates:", top_available_candidates)

ensemble_results = {}

# Make sure CV columns exist
for col in ["CV_MAE", "CV_RMSE", "CV_R2"]:
    if col not in results_df.columns:
        results_df[col] = np.nan

if len(top_available_candidates) >= 2:

    voting_estimators = []

    for name in top_available_candidates:
        preprocessor, model = models[name]

        voting_estimators.append((name, Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ])))

    # Fast Voting Ensemble only
    voting_model = VotingRegressor(
        estimators=voting_estimators,
        n_jobs=-1
    )

    print("\nTraining ensemble: Voting_Ensemble")

    try:
        voting_model.fit(X_train, y_train)
        y_pred = voting_model.predict(X_test)

        mae, rmse, r2, mape = regression_metrics(y_test, y_pred)

        ensemble_results["Voting_Ensemble"] = voting_model

        new_row = pd.DataFrame([{
            "Model": "Voting_Ensemble",
            "CV_MAE": np.nan,
            "CV_RMSE": np.nan,
            "CV_R2": np.nan,
            "Test_MAE": mae,
            "Test_RMSE": rmse,
            "Test_R2": r2,
            "Test_MAPE": mape
        }])

        results_df = pd.concat([results_df, new_row], ignore_index=True)

    except Exception as e:
        print("Voting Ensemble failed.")
        print("Error:", e)

else:
    print("Not enough strong models available for ensemble creation.")

results_df = results_df.sort_values(by="Test_RMSE", ascending=True).reset_index(drop=True)
display(results_df)

results_df.to_csv("model_comparison_results_with_fast_ensemble_ev_ufug.csv", index=False)
print("Saved: model_comparison_results_with_fast_ensemble_ev_ufug.csv")
'''

## 15. Select Best Model

In [ ]:
# Test-set prediction and residual analysis for the CV-selected model.
residuals = np.asarray(y_test) - np.asarray(best_test_pred)
prediction_errors = pd.DataFrame({
    "actual_ufug": np.asarray(y_test),
    "predicted_ufug": np.asarray(best_test_pred),
    "residual_actual_minus_predicted": residuals
})
prediction_errors.to_csv("test_set_predictions_and_residuals.csv", index=False)

lo = float(min(np.min(y_test), np.min(best_test_pred)))
hi = float(max(np.max(y_test), np.max(best_test_pred)))
plt.figure(figsize=(6.5, 5.5))
plt.scatter(y_test, best_test_pred, alpha=0.65)
plt.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1.5, label="Perfect prediction")
plt.xlabel("Actual UFUG (gal/100 mi)")
plt.ylabel("Predicted UFUG (gal/100 mi)")
plt.title(f"Actual vs Predicted UFUG: {best_model_name}")
plt.legend()
plt.tight_layout()
plt.savefig("actual_vs_predicted_ufug.png", dpi=300, bbox_inches="tight")
plt.savefig("actual_vs_predicted_ufug.pdf", bbox_inches="tight")
plt.show()

plt.figure(figsize=(6.5, 5.0))
plt.scatter(best_test_pred, residuals, alpha=0.65)
plt.axhline(0, color="black", linestyle="--", linewidth=1.2)
plt.xlabel("Predicted UFUG (gal/100 mi)")
plt.ylabel("Residual (actual − predicted)")
plt.title(f"Residuals vs Predicted UFUG: {best_model_name}")
plt.tight_layout()
plt.savefig("residuals_vs_predicted_ufug.png", dpi=300, bbox_inches="tight")
plt.savefig("residuals_vs_predicted_ufug.pdf", bbox_inches="tight")
plt.show()

plt.figure(figsize=(6.5, 5.0))
plt.hist(residuals, bins=30, edgecolor="black")
plt.axvline(0, color="black", linestyle="--", linewidth=1.2)
plt.xlabel("Residual (gal/100 mi)")
plt.ylabel("Count")
plt.title(f"Test Residual Distribution: {best_model_name}")
plt.tight_layout()
plt.savefig("residual_distribution_ufug.png", dpi=300, bbox_inches="tight")
plt.savefig("residual_distribution_ufug.pdf", bbox_inches="tight")
plt.show()


## 16. Permutation Feature Importance

Permutation importance works for any trained model and is suitable for explaining global feature importance.

In [ ]:
try:
    perm = permutation_importance(
        best_model,
        X_test,
        y_test,
        n_repeats=10,
        random_state=RANDOM_STATE,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )

    importance_df = pd.DataFrame({
        "Feature": X_test.columns,
        "Importance_Mean": perm.importances_mean,
        "Importance_STD": perm.importances_std
    }).sort_values(by="Importance_Mean", ascending=False)

    display(importance_df.head(20))
    importance_df.to_csv("permutation_feature_importance.csv", index=False)

    plt.figure(figsize=(10, 6))
    top_imp = importance_df.head(15).sort_values("Importance_Mean")
    plt.barh(top_imp["Feature"], top_imp["Importance_Mean"])
    plt.xlabel("Permutation Importance")
    plt.title("Top Feature Importance")
    plt.tight_layout()
    plt.show()

except Exception as e:
    print("Permutation importance failed.")
    print("Error:", e)

## 17. SHAP Explainability

This section provides SHAP explainability.

For complex ensemble models, SHAP can be slow. The notebook uses a small sample to keep runtime reasonable.

In [ ]:
import shap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.pipeline import Pipeline

# --------------------------------------------------------
# Use best individual model for SHAP, not ensemble
# Extra Trees was selected because it is accurate and faster
# --------------------------------------------------------

#shap_model_name = best_model_name
shap_model_name = best_model_name

if shap_model_name not in models:
    raise ValueError(f"{shap_model_name} not found in models dictionary.")

preprocessor, base_model = models[shap_model_name]

shap_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", base_model)
])

print(f"Training SHAP model: {shap_model_name}")
shap_pipeline.fit(X_train, y_train)

# --------------------------------------------------------
# Use small sample to reduce runtime
# --------------------------------------------------------
X_shap_sample = X_test.sample(
    min(100, len(X_test)),
    random_state=RANDOM_STATE
)

In [ ]:
# --------------------------------------------------------
# Transform categorical + numerical features into numeric form
# --------------------------------------------------------
X_shap_transformed = shap_pipeline.named_steps["preprocessor"].transform(X_shap_sample)

if sparse.issparse(X_shap_transformed):
    X_shap_transformed = X_shap_transformed.toarray()

# --------------------------------------------------------
# Get transformed feature names
# --------------------------------------------------------
try:
    feature_names = shap_pipeline.named_steps["preprocessor"].get_feature_names_out()
except:
    feature_names = [f"feature_{i}" for i in range(X_shap_transformed.shape[1])]

X_shap_transformed_df = pd.DataFrame(
    X_shap_transformed,
    columns=feature_names
)

In [ ]:
'''
# --------------------------------------------------------
# SHAP TreeExplainer for tree-based model
# --------------------------------------------------------
explainer = shap.TreeExplainer(shap_pipeline.named_steps["model"])
shap_values = explainer.shap_values(X_shap_transformed_df)

print("SHAP explanation completed successfully.")

# --------------------------------------------------------
# SHAP Summary Plot
# --------------------------------------------------------
shap.summary_plot(
    shap_values,
    X_shap_transformed_df,
    max_display=20,
    show=False
)

plt.title(f"SHAP Feature Importance - {best_model_name} Model")
plt.tight_layout()
plt.savefig(f"shap_summary_{best_model_name}_ev_ufug.png", dpi=300, bbox_inches="tight")
plt.show()
'''
# --------------------------------------------------------
# SHAP TreeExplainer for tree-based model - Fixed Version
# --------------------------------------------------------

# Create background data using the SAME fitted preprocessor
X_background_sample = X_train.sample(
    min(200, len(X_train)),
    random_state=RANDOM_STATE
)

X_background_transformed = shap_pipeline.named_steps["preprocessor"].transform(X_background_sample)

if sparse.issparse(X_background_transformed):
    X_background_transformed = X_background_transformed.toarray()

# Use interventional TreeExplainer to avoid additivity mismatch
explainer = shap.TreeExplainer(
    shap_pipeline.named_steps["model"],
    data=X_background_transformed,
    feature_perturbation="interventional"
)

# Disable additivity check because sklearn pipeline + one-hot encoding can cause mismatch
shap_values = explainer.shap_values(
    X_shap_transformed_df,
    check_additivity=False
)

print("SHAP explanation completed successfully.")

# --------------------------------------------------------
# SHAP Summary Plot
# --------------------------------------------------------
shap.summary_plot(
    shap_values,
    X_shap_transformed_df,
    max_display=20,
    show=False
)

plt.title(f"SHAP Feature Importance - {shap_model_name} Model")
plt.tight_layout()
plt.savefig(f"shap_summary_{shap_model_name}_ev_ufug.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# --------------------------------------------------------
# SHAP Bar Plot
# --------------------------------------------------------
shap.summary_plot(
    shap_values,
    X_shap_transformed_df,
    plot_type="bar",
    max_display=20,
    show=False
)

'''
plt.title(f"SHAP Mean Feature Importance - {best_model_name} Model")
plt.tight_layout()
plt.savefig(f"shap_bar_{best_model_name}_ev_ufug.png", dpi=300, bbox_inches="tight")
plt.show()
'''
plt.title(f"SHAP Mean Feature Importance - {shap_model_name} Model")
plt.tight_layout()
plt.savefig(f"shap_bar_{shap_model_name}_ev_ufug.png", dpi=300, bbox_inches="tight")

## 18. Optional LIME Local Explanation

LIME explains one individual prediction. This is useful for showing a sample vehicle-level explanation in the paper.

In [ ]:

try:
    from lime.lime_tabular import LimeTabularExplainer

    X_train_for_lime = X_train.copy()
    X_test_for_lime = X_test.copy()

    # Convert categorical variables to category codes only for LIME.
    lime_train = X_train_for_lime.copy()
    lime_test = X_test_for_lime.copy()

    for col in lime_train.select_dtypes(exclude=[np.number]).columns:
        categories = pd.concat([lime_train[col], lime_test[col]], axis=0).astype(str).unique()
        mapping = {cat: i for i, cat in enumerate(categories)}
        lime_train[col] = lime_train[col].astype(str).map(mapping)
        lime_test[col] = lime_test[col].astype(str).map(mapping)

    lime_train = lime_train.fillna(lime_train.median(numeric_only=True))
    lime_test = lime_test.fillna(lime_train.median(numeric_only=True))

    lime_explainer = LimeTabularExplainer(
        training_data=lime_train.values,
        feature_names=lime_train.columns.tolist(),
        mode="regression",
        random_state=RANDOM_STATE
    )

    # Prediction function using original best_model requires converting back column names.
    def lime_predict(arr):
        arr_df = pd.DataFrame(arr, columns=lime_train.columns)
        # LIME uses encoded categories, so this is approximate.
        # For final paper, SHAP/permutation importance is preferred.
        return best_model.predict(arr_df)

    idx = 0
    lime_exp = lime_explainer.explain_instance(
        lime_test.iloc[idx].values,
        lime_predict,
        num_features=10
    )

    print("LIME explanation for one sample:")
    display(lime_exp.as_list())

except Exception as e:
    print("LIME explanation failed.")
    print("Error:", e)
    print("For the final paper, use SHAP/permutation importance if LIME is not stable.")



## 19. EV Savings and Conversion Priority Decision System

This module converts regression predictions into vehicle replacement decisions.

You can adjust these assumptions according to your country, paper, or scenario.

In [ ]:
# =============================
# Scenario assumptions
# =============================
DAILY_MILES = 80
WORKING_DAYS_PER_YEAR = 300
CITY_DRIVING_SHARE = 0.70

GAS_PRICE_PER_GALLON = 3.50
ELECTRICITY_PRICE_PER_KWH = 0.15
EV_KWH_PER_MILE = 0.30

NET_EV_REPLACEMENT_COST = 30000
TOTAL_REPLACEMENT_BUDGET = 300000

ANNUAL_MILES = DAILY_MILES * WORKING_DAYS_PER_YEAR
ANNUAL_URBAN_MILES = ANNUAL_MILES * CITY_DRIVING_SHARE

print("Annual miles:", ANNUAL_MILES)
print("Annual urban miles:", ANNUAL_URBAN_MILES)

priority_data = data_raw_for_ranking.loc[X.index].copy()
priority_features = X.copy()

# added based on chat suggestions
#This stops unrealistic vehicles like Ferrari, Lamborghini, and Bugatti from appearing as fleet replacement candidates.
FLEET_ONLY_RANKING = True

if FLEET_ONLY_RANKING and vehicle_class_col and vehicle_class_col in priority_data.columns:
    fleet_keywords = [
        "suv", "pickup", "van", "minivan",
        "station wagon", "wagon",
        "truck"
    ]

    class_text = (
        priority_data[vehicle_class_col]
        .astype(str)
        .str.lower()
        .str.replace("_", " ", regex=False)
        .str.replace("-", " ", regex=False)
    )

    fleet_pattern = "|".join([re.escape(k) for k in fleet_keywords])
    fleet_mask = class_text.str.contains(fleet_pattern, na=False)

    if fleet_mask.sum() > 0:
        priority_data = priority_data.loc[fleet_mask].copy()
        priority_features = priority_features.loc[priority_data.index].copy()
        print("Fleet-only ranking enabled.")
        print("Fleet-like vehicles kept:", len(priority_data))
    else:
        print("No fleet-like vehicles found. Continuing with all vehicle classes.")

priority_data["predicted_ufug"] = best_model.predict(priority_features)

# Predicted extra gallons wasted because of urban/stop-go penalty
priority_data["ufug_associated_annual_quantity_gal"] = (
    priority_data["predicted_ufug"] * ANNUAL_URBAN_MILES / 100
)

priority_data["ufug_associated_annual_cost_estimate"] = (
    priority_data["ufug_associated_annual_quantity_gal"] * GAS_PRICE_PER_GALLON
)

# Total gasoline operating cost estimate using actual city fuel rate if available
priority_data["estimated_gas_city_gallons_per_year"] = (
    priority_data["city_gal_per_100mi"] * ANNUAL_MILES / 100
)

priority_data["estimated_gas_city_cost_per_year"] = (
    priority_data["estimated_gas_city_gallons_per_year"] * GAS_PRICE_PER_GALLON
)

priority_data["estimated_ev_energy_cost_per_year"] = (
    ANNUAL_MILES * EV_KWH_PER_MILE * ELECTRICITY_PRICE_PER_KWH
)

priority_data["estimated_annual_operating_cost_savings"] = (
    priority_data["estimated_gas_city_cost_per_year"] - priority_data["estimated_ev_energy_cost_per_year"]
)

# CO2 reduction proxy if CO2 column exists
if co2_col and co2_col in priority_data.columns:
    priority_data[co2_col] = pd.to_numeric(priority_data[co2_col], errors="coerce")
    # g/km to kg/year: annual miles -> km
    annual_km = ANNUAL_MILES * 1.60934
    priority_data["estimated_annual_vehicle_co2_emissions"] = priority_data[co2_col] * annual_km / 1000
else:
    priority_data["estimated_annual_vehicle_co2_emissions"] = np.nan

# EV suitability score based on vehicle class.
# This is a simple paper-friendly rule. You can modify it later.
def ev_suitability_score(row):
    text = ""
    if vehicle_class_col and vehicle_class_col in row.index:
        text += str(row[vehicle_class_col]).lower()
    if "compact" in text or "subcompact" in text or "mid_size" in text or "midsize" in text or "station_wagon" in text:
        return 1.00
    if "suv" in text or "pickup" in text or "van" in text:
        return 0.80
    if "truck" in text or "special" in text:
        return 0.60
    return 0.75

priority_data["ev_suitability_score"] = priority_data.apply(ev_suitability_score, axis=1)

def minmax(series):
    series = pd.to_numeric(series, errors="coerce")
    if series.max() == series.min():
        return pd.Series(np.ones(len(series)), index=series.index)
    return (series - series.min()) / (series.max() - series.min())

priority_data["waste_score_norm"] = minmax(priority_data["predicted_ufug"])
priority_data["savings_score_norm"] = minmax(priority_data["estimated_annual_operating_cost_savings"].clip(lower=0))

if priority_data["estimated_annual_vehicle_co2_emissions"].notna().sum() > 0:
    priority_data["co2_score_norm"] = minmax(priority_data["estimated_annual_vehicle_co2_emissions"])
else:
    priority_data["co2_score_norm"] = 0.50

priority_data["ev_priority_score"] = (
    0.40 * priority_data["waste_score_norm"] +
    0.30 * priority_data["savings_score_norm"] +
    0.20 * priority_data["co2_score_norm"] +
    0.10 * priority_data["ev_suitability_score"]
)

# Replacement decision based on percentile
p80 = priority_data["ev_priority_score"].quantile(0.80)
p60 = priority_data["ev_priority_score"].quantile(0.60)
p40 = priority_data["ev_priority_score"].quantile(0.40)

def replacement_decision(score):
    if score >= p80:
        return "Immediate EV replacement candidate"
    elif score >= p60:
        return "High priority replacement"
    elif score >= p40:
        return "Medium priority: monitor and replace later"
    else:
        return "Low priority: keep gasoline vehicle now"

priority_data["conversion_priority"] = priority_data["ev_priority_score"].apply(replacement_decision)

# Budget-optimized schedule
max_replacements = int(TOTAL_REPLACEMENT_BUDGET // NET_EV_REPLACEMENT_COST)

priority_ranked = priority_data.sort_values(by="ev_priority_score", ascending=False).reset_index(drop=True)
priority_ranked["priority_rank"] = np.arange(1, len(priority_ranked) + 1)
priority_ranked["budget_selected"] = priority_ranked["priority_rank"] <= max_replacements

print("Maximum EV replacements under budget:", max_replacements)

# Useful columns for display
display_cols = [
    "priority_rank",
    "predicted_ufug",
    "ufug_associated_annual_quantity_gal",
    "ufug_associated_annual_cost_estimate",
    "estimated_annual_operating_cost_savings",
    "estimated_annual_vehicle_co2_emissions",
    "ev_suitability_score",
    "ev_priority_score",
    "conversion_priority",
    "budget_selected"
]

# Add identity columns if available
identity_cols = [c for c in [year_col, make_col, model_col, vehicle_class_col, engine_col, cylinder_col, fuel_type_col, transmission_col] if c and c in priority_ranked.columns]
display_cols = identity_cols + display_cols

display(priority_ranked[display_cols].head(30))

#priority_ranked[display_cols].to_csv("ev_conversion_priority_results.csv", index=False)
#print("Saved: ev_conversion_priority_results.csv")

In [ ]:
# Corrected scenario calculations: consistent 70:30 city/highway driving mix.
# These are scenario estimates, not measured or guaranteed savings.
priority_data["estimated_annual_gasoline_gallons"] = (
    (CITY_DRIVING_SHARE * priority_data["city_gal_per_100mi"]) +
    ((1 - CITY_DRIVING_SHARE) * priority_data["hwy_gal_per_100mi"])
) * ANNUAL_MILES / 100
priority_data["estimated_annual_gasoline_cost"] = (
    priority_data["estimated_annual_gasoline_gallons"] * GAS_PRICE_PER_GALLON
)
priority_data["estimated_annual_ev_electricity_kwh"] = ANNUAL_MILES * EV_KWH_PER_MILE
priority_data["estimated_annual_ev_electricity_cost"] = (
    priority_data["estimated_annual_ev_electricity_kwh"] * ELECTRICITY_PRICE_PER_KWH
)
priority_data["estimated_annual_operating_cost_savings"] = (
    priority_data["estimated_annual_gasoline_cost"] -
    priority_data["estimated_annual_ev_electricity_cost"]
)

# Scenario-based vehicle CO2 emissions. This is tailpipe vehicle emissions, not lifecycle emissions.
if co2_col and co2_col in priority_data.columns:
    priority_data[co2_col] = pd.to_numeric(priority_data[co2_col], errors="coerce")
    annual_km = ANNUAL_MILES * 1.60934
    priority_data["estimated_annual_vehicle_co2_emissions"] = priority_data[co2_col] * annual_km / 1000
else:
    priority_data["estimated_annual_vehicle_co2_emissions"] = np.nan

def minmax_safe(series):
    s = pd.to_numeric(series, errors="coerce")
    lo, hi = s.min(), s.max()
    if pd.isna(lo) or pd.isna(hi) or hi == lo:
        return pd.Series(0.5, index=s.index)
    return (s - lo) / (hi - lo)

priority_data["ufug_score_norm"] = minmax_safe(priority_data["predicted_ufug"])
priority_data["savings_score_norm"] = minmax_safe(priority_data["estimated_annual_operating_cost_savings"].clip(lower=0))
if priority_data["estimated_annual_vehicle_co2_emissions"].notna().any():
    priority_data["co2_score_norm"] = minmax_safe(priority_data["estimated_annual_vehicle_co2_emissions"])
else:
    priority_data["co2_score_norm"] = 0.5

# Baseline weights are explicit, non-negative, and sum to one.
BASELINE_WEIGHTS = {"ufug": 0.40, "savings": 0.30, "co2": 0.20, "suitability": 0.10}
priority_data["ev_priority_score"] = (
    BASELINE_WEIGHTS["ufug"] * priority_data["ufug_score_norm"] +
    BASELINE_WEIGHTS["savings"] * priority_data["savings_score_norm"] +
    BASELINE_WEIGHTS["co2"] * priority_data["co2_score_norm"] +
    BASELINE_WEIGHTS["suitability"] * priority_data["ev_suitability_score"]
)
priority_ranked = priority_data.sort_values("ev_priority_score", ascending=False).reset_index(drop=True)
priority_ranked["priority_rank"] = np.arange(1, len(priority_ranked) + 1)
max_replacements = int(TOTAL_REPLACEMENT_BUDGET // NET_EV_REPLACEMENT_COST)
priority_ranked["budget_selected"] = priority_ranked["priority_rank"] <= max_replacements

# Weight sensitivity: each alternative remains non-negative and sums to 1.
weight_scenarios = {
    "baseline": {"ufug": .40, "savings": .30, "co2": .20, "suitability": .10},
    "savings_focused": {"ufug": .25, "savings": .45, "co2": .20, "suitability": .10},
    "co2_focused": {"ufug": .25, "savings": .25, "co2": .40, "suitability": .10},
    "ufug_focused": {"ufug": .55, "savings": .20, "co2": .15, "suitability": .10}
}
top_k = min(10, len(priority_ranked))
base_top = set(priority_ranked.head(top_k).index)
sensitivity_rows = []
for scenario, w in weight_scenarios.items():
    assert all(v >= 0 for v in w.values()) and abs(sum(w.values()) - 1.0) < 1e-9
    scores = (
        w["ufug"] * priority_data["ufug_score_norm"] +
        w["savings"] * priority_data["savings_score_norm"] +
        w["co2"] * priority_data["co2_score_norm"] +
        w["suitability"] * priority_data["ev_suitability_score"]
    )
    selected_idx = set(scores.sort_values(ascending=False).head(top_k).index)
    sensitivity_rows.append({
        "scenario": scenario, **{f"weight_{k}": v for k,v in w.items()},
        "top_k_overlap_count_vs_baseline": len(selected_idx & base_top),
        "top_k_overlap_fraction_vs_baseline": len(selected_idx & base_top) / max(top_k,1),
        "selected_vehicles_estimated_annual_operating_cost_savings": float(priority_data.loc[list(selected_idx), "estimated_annual_operating_cost_savings"].sum()),
        "selected_vehicles_estimated_annual_vehicle_co2_emissions": float(priority_data.loc[list(selected_idx), "estimated_annual_vehicle_co2_emissions"].sum(min_count=1))
    })
pd.DataFrame(sensitivity_rows).to_csv("ev_priority_weight_sensitivity.csv", index=False)
priority_ranked.to_csv("ev_conversion_priority_results.csv", index=False)
print("Saved corrected scenario costs and ev_priority_weight_sensitivity.csv")
print("Annual gasoline costs use the configured 70:30 city/highway blend.")


## 20. Top 20% EV Conversion Candidates

In [ ]:
top_20_percent_cutoff = int(np.ceil(0.20 * len(priority_ranked)))
top_20_candidates = priority_ranked.head(top_20_percent_cutoff)

print("Top 20% candidate count:", len(top_20_candidates))
display(top_20_candidates[display_cols].head(30))

top_20_candidates[display_cols].to_csv("top_20_percent_ev_conversion_candidates.csv", index=False)
print("Saved: top_20_percent_ev_conversion_candidates.csv")

## 21. Save Best Model

In [ ]:
import joblib

model_filename = "best_ufug_model.pkl"
joblib.dump(best_model, model_filename)

metadata = {
    "best_model_name": best_model_name,
    "target": TARGET,
    "city_column": city_col,
    "highway_column": hwy_col,
    "leakage_safe_mode": LEAKAGE_SAFE_MODE,
    "feature_columns": list(X.columns),
    "daily_miles": DAILY_MILES,
    "working_days_per_year": WORKING_DAYS_PER_YEAR,
    "city_driving_share": CITY_DRIVING_SHARE,
    "gas_price_per_gallon": GAS_PRICE_PER_GALLON,
    "electricity_price_per_kwh": ELECTRICITY_PRICE_PER_KWH,
    "ev_kwh_per_mile": EV_KWH_PER_MILE,
    "net_ev_replacement_cost": NET_EV_REPLACEMENT_COST,
    "total_replacement_budget": TOTAL_REPLACEMENT_BUDGET
}

with open("project_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Saved:", model_filename)
print("Saved: project_metadata.json")

## 22. Download Output Files from Colab

In [ ]:

from google.colab import files

output_files = [
    "model_comparison_results_ev_ufug.csv",
    "model_comparison_results_with_ensembles_ev_ufug.csv",
    "permutation_feature_importance.csv",
    "ev_conversion_priority_results.csv",
    "top_20_percent_ev_conversion_candidates.csv",
    "best_ufug_model.pkl",
    "project_metadata.json"
]

for file in output_files:
    if os.path.exists(file):
        print("Downloading:", file)
        files.download(file)
    else:
        print("File not found:", file)


## 23. Research Paper Result Summary Template

After running the notebook, use this table in your paper:

- Best model name
- MAE
- RMSE
- R²
- Important features from SHAP/permutation importance
- Top 20% replacement candidates
- Estimated annual savings
- Budget-selected replacement list

### Important wording for the paper
Use this sentence:

> This study estimates Urban Fuel-Use Gap (UFUG) using the city–highway fuel-consumption gap as a proxy. The proposed model does not directly measure real UFUG from onboard sensors; instead, it provides a vehicle-level decision-support estimate for EV conversion prioritization.

## 24. Additional Evaluation and Sensitivity Analysis

This section creates a Random Forest learning curve only when the selected model is a Random Forest, evaluates a grouped split that holds out vehicle-model groups when the required identifier is available, and saves priority-weight sensitivity results. Metrics and figures must be regenerated by running the notebook with the dataset present.


In [ ]:
# Additional research evaluation: learning curve and grouped unseen-model split.
from sklearn.model_selection import learning_curve, GroupShuffleSplit
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor

# Learning curve is meaningful for the selected Random Forest. Do not fabricate epoch curves.
selected_estimator = best_model.named_steps["model"]
if isinstance(selected_estimator, RandomForestRegressor):
    sizes, train_scores, val_scores = learning_curve(
        best_model, X_train, y_train, cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        train_sizes=np.linspace(0.2, 1.0, 5), scoring="neg_root_mean_squared_error",
        n_jobs=1, shuffle=True, random_state=RANDOM_STATE
    )
    curve_df = pd.DataFrame({
        "training_examples": sizes,
        "training_rmse_mean": -train_scores.mean(axis=1),
        "training_rmse_std": train_scores.std(axis=1),
        "validation_rmse_mean": -val_scores.mean(axis=1),
        "validation_rmse_std": val_scores.std(axis=1)
    })
    curve_df.to_csv("learning_curve_random_forest.csv", index=False)
    plt.figure(figsize=(7, 5))
    plt.plot(sizes, curve_df["training_rmse_mean"], marker="o", label="Training RMSE")
    plt.plot(sizes, curve_df["validation_rmse_mean"], marker="o", label="Cross-validation RMSE")
    plt.xlabel("Training-set size")
    plt.ylabel("RMSE (gal/100 mi)")
    plt.title("Random Forest Learning Curve")
    plt.legend()
    plt.tight_layout()
    plt.savefig("learning_curve_random_forest.png", dpi=300, bbox_inches="tight")
    plt.savefig("learning_curve_random_forest.pdf", bbox_inches="tight")
    plt.show()
else:
    print(f"Selected model is {type(selected_estimator).__name__}, not RandomForestRegressor.")
    print("No Random Forest learning curve was generated. The saved test residual plots are the appropriate error analysis for the selected model.")

# Grouped split: no vehicle_model_group may occur in both train and test.
# Prefer the model-identity grouping column retained in the cleaned data.
if "vehicle_model_group" in data.columns:
    groups_all = data.loc[X.index, "vehicle_model_group"].astype("string").fillna("__MISSING_MODEL_GROUP__")
elif model_col and model_col in data.columns:
    groups_all = (
        data.loc[X.index, make_col].astype("string").fillna("__MISSING_MAKE__") + "::" +
        data.loc[X.index, model_col].astype("string").fillna("__MISSING_MODEL__")
    ) if make_col and make_col in data.columns else data.loc[X.index, model_col].astype("string").fillna("__MISSING_MODEL__")
else:
    groups_all = None

if groups_all is not None and groups_all.nunique() >= 2:
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    g_train_idx, g_test_idx = next(gss.split(X, y, groups=groups_all))
    Xg_train, Xg_test = X.iloc[g_train_idx], X.iloc[g_test_idx]
    yg_train_raw, yg_test_raw = y.iloc[g_train_idx], y.iloc[g_test_idx]
    # Derive target caps from grouped training data only.
    q1g, q3g = yg_train_raw.quantile(.25), yg_train_raw.quantile(.75)
    iqrg = q3g - q1g
    lower_g, upper_g = max(0.0, q1g - 1.5*iqrg), q3g + 1.5*iqrg
    yg_train = yg_train_raw.clip(lower=lower_g, upper=upper_g) if TARGET_CAP_IQR else yg_train_raw
    yg_test = yg_test_raw.clip(lower=lower_g, upper=upper_g) if TARGET_CAP_IQR else yg_test_raw
    grouped_model = clone(best_model)
    grouped_model.fit(Xg_train, yg_train)
    grouped_pred = grouped_model.predict(Xg_test)
    group_metrics = pd.DataFrame([{
        "split": "unseen_vehicle_model_group",
        "model_selected_by_random_split_cv": best_model_name,
        "MAE": mean_absolute_error(yg_test, grouped_pred),
        "RMSE": np.sqrt(mean_squared_error(yg_test, grouped_pred)),
        "R2": r2_score(yg_test, grouped_pred),
        "train_rows": len(g_train_idx),
        "test_rows": len(g_test_idx),
        "train_unique_groups": groups_all.iloc[g_train_idx].nunique(),
        "test_unique_groups": groups_all.iloc[g_test_idx].nunique(),
        "group_overlap": len(set(groups_all.iloc[g_train_idx]) & set(groups_all.iloc[g_test_idx]))
    }])
    group_metrics.to_csv("unseen_vehicle_model_evaluation.csv", index=False)
    display(group_metrics)
    assert group_metrics.loc[0, "group_overlap"] == 0, "Model-group leakage detected in grouped split."
else:
    print("Unseen-model experiment skipped: a usable vehicle-model grouping column was not available.")

print("Outputs saved: learning curve (if supported), grouped evaluation (if possible), test predictions/residuals, and priority sensitivity.")
